# Notebook 11 — Interactive Station Map
Three-layer Folium map of all well-covered AWV stations:
- **Layer 1**: Station type (commuter / mixed / leisure)
- **Layer 2**: Infrastructure investment priority
- **Layer 3**: Cycling volume

Output: `outputs/interactive_map.html`

## 0. Imports & Load Data

In [4]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "folium"])

import folium
import numpy as np
import pandas as pd
from pathlib import Path

ROOT      = Path().resolve().parent
PROCESSED = ROOT / "data" / "processed"
OUTPUT    = ROOT / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)

panel      = pd.read_parquet(PROCESSED / "analysis_panel.parquet")
site_class = pd.read_parquet(PROCESSED / "site_classification.parquet")
resilience = pd.read_parquet(PROCESSED / "station_resilience.parquet")

## 1. Build Station-Level Dataset

In [5]:
# One row per well-covered station with all needed attributes
stations = (
    panel[panel["low_coverage"] == False]
    .groupby(["site ID", "naam", "gemeente", "lat", "long"])
    ["aantal"].mean()
    .reset_index()
    .rename(columns={"aantal": "avg_daily"})
)

stations = stations.merge(
    site_class[["site ID", "site_type", "commute_score",
                "weekday_weekend_ratio", "avg_peak_ratio"]],
    on="site ID", how="left"
)

stations = stations.merge(
    resilience[["site ID", "resilience_score", "priority_rank",
                "resilience_rain", "resilience_cold",
                "resilience_wind"]],
    on="site ID", how="left"
)

stations = stations.dropna(subset=["lat", "long"])
print(f"Stations to plot: {len(stations)}")
print(stations[["naam", "lat", "long", "site_type",
                "avg_daily", "priority_rank"]].head())

Stations to plot: 133
           naam        lat      long site_type   avg_daily  priority_rank
0      Machelen  50.916183  4.456122     mixed  395.821167           64.0
1  Brasschaat 2  51.275120  4.471690  commuter  822.481750          107.0
2  Brasschaat 1  51.275030  4.472220  commuter  759.341553          118.0
3       Balen 1  51.160230  5.190110   leisure  186.947998           45.0
4       Balen 2  51.160180  5.190030   leisure  195.414230           38.0


## 2. Color & Size Configuration

In [6]:
COLORS = {
    "commuter": "#2196F3",
    "mixed":    "#FFC107",
    "leisure":  "#F44336",
    "unknown":  "#9E9E9E",
}

def get_radius(avg_daily):
    """Scale circle size by sqrt of daily volume, capped 5-25."""
    return max(5, min(np.sqrt(avg_daily) / 3, 25))

def get_priority_color(rank, total=132):
    """Red = high priority (rank 1), Green = low priority. Gray if NaN."""
    if pd.isna(rank):
        return "#9E9E9E"
    pct = 1 - (rank - 1) / (total - 1)
    r = int(255 * pct)
    g = int(255 * (1 - pct))
    return f"#{r:02x}{g:02x}00"

def fmt_rank(val):
    """Safely convert rank to int string, returning N/A for NaN."""
    return "N/A" if pd.isna(val) else str(int(val))

## 3. Create Base Map

In [7]:
m = folium.Map(
    location=[51.0, 4.2],
    zoom_start=9,
    tiles="CartoDB positron"
)

## 4. Layer 1: Station Type View (default, visible)

In [8]:
type_layer = folium.FeatureGroup(
    name="Station type (commuter / mixed / leisure)",
    show=True
)

for _, row in stations.iterrows():
    color  = COLORS.get(row["site_type"], COLORS["unknown"])
    radius = get_radius(row["avg_daily"])

    popup_html = f"""
    <div style="font-family:Arial; font-size:13px; width:230px;">
        <h4 style="margin:0 0 4px 0; color:{color};">
            {row['naam']}
        </h4>
        <hr style="margin:4px 0;">
        <table style="width:100%; border-collapse:collapse;">
            <tr><td><b>Gemeente</b></td>
                <td>{row['gemeente']}</td></tr>
            <tr><td><b>Type</b></td>
                <td style="color:{color}"><b>{row['site_type']}</b></td></tr>
            <tr><td><b>Avg daily cyclists</b></td>
                <td>{row['avg_daily']:.0f}</td></tr>
            <tr><td><b>Weekday/weekend ratio</b></td>
                <td>{row['weekday_weekend_ratio']:.2f}</td></tr>
            <tr><td><b>Peak hour ratio</b></td>
                <td>{row['avg_peak_ratio']:.2f}</td></tr>
            <tr><td><b>Resilience score</b></td>
                <td>{row['resilience_score']:.3f}</td></tr>
            <tr><td><b>Rain resilience</b></td>
                <td>{row['resilience_rain']:.3f}</td></tr>
            <tr><td><b>Cold resilience</b></td>
                <td>{row['resilience_cold']:.3f}</td></tr>
            <tr><td><b>Priority rank</b></td>
                <td>#{fmt_rank(row['priority_rank'])}/132</td></tr>
        </table>
    </div>
    """

    folium.CircleMarker(
        location=[row["lat"], row["long"]],
        radius=radius,
        color="white",
        weight=0.8,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=(
            f"{row['naam']} | {row['site_type']} | "
            f"{row['avg_daily']:.0f} cyclists/day"
        )
    ).add_to(type_layer)

type_layer.add_to(m)

## 5. Layer 2: Investment Priority View (hidden by default)

In [9]:
priority_layer = folium.FeatureGroup(
    name="Infrastructure investment priority",
    show=False
)

for _, row in stations.iterrows():
    color = get_priority_color(row["priority_rank"])

    popup_html = f"""
    <div style="font-family:Arial; font-size:13px; width:220px;">
        <h4 style="margin:0 0 4px 0;">
            #{fmt_rank(row['priority_rank'])} {row['naam']}
        </h4>
        <hr style="margin:4px 0;">
        <b>Gemeente:</b> {row['gemeente']}<br>
        <b>Type:</b> {row['site_type']}<br>
        <b>Avg daily cyclists:</b> {row['avg_daily']:.0f}<br>
        <b>Resilience score:</b> {row['resilience_score']:.3f}<br>
        <b>Priority rank:</b> #{fmt_rank(row['priority_rank'])}/132<br>
        <small style="color:#666;">
            Red = most urgent investment needed<br>
            Green = already resilient
        </small>
    </div>
    """

    folium.CircleMarker(
        location=[row["lat"], row["long"]],
        radius=9,
        color="white",
        weight=0.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=(
            f"Priority #{fmt_rank(row['priority_rank'])} | "
            f"{row['naam']} | resilience={row['resilience_score']:.2f}"
        )
    ).add_to(priority_layer)

priority_layer.add_to(m)

## 6. Layer 3: Volume View (hidden by default)

In [10]:
volume_layer = folium.FeatureGroup(
    name="Cycling volume (size = daily cyclists)",
    show=False
)

max_vol = stations["avg_daily"].max()
for _, row in stations.iterrows():
    intensity = row["avg_daily"] / max_vol

    folium.CircleMarker(
        location=[row["lat"], row["long"]],
        radius=get_radius(row["avg_daily"]),
        color="white",
        weight=0.5,
        fill=True,
        fill_color="#1565C0",
        fill_opacity=0.4 + 0.5 * intensity,
        tooltip=(
            f"{row['naam']} | "
            f"{row['avg_daily']:.0f} cyclists/day"
        ),
        popup=folium.Popup(
            f"<b>{row['naam']}</b><br>"
            f"Avg daily: {row['avg_daily']:.0f}<br>"
            f"Gemeente: {row['gemeente']}",
            max_width=200
        )
    ).add_to(volume_layer)

volume_layer.add_to(m)

## 7. Special Marker: Leuven Totem

In [11]:
leuven_row = stations[stations["site ID"] == 107]
if len(leuven_row) > 0:
    r = leuven_row.iloc[0]
    folium.Marker(
        location=[r["lat"], r["long"]],
        tooltip=f"Leuven Totem — #1 station ({r['avg_daily']:.0f}/day)",
        popup=folium.Popup(
            f"""
            <div style="font-family:Arial; width:200px;">
                <h4 style="color:#1565C0;">Leuven Totem</h4>
                <b>#1 station by volume</b><br>
                Avg daily: {r['avg_daily']:.0f} cyclists<br>
                Type: {r['site_type']}<br>
                Resilience: {r['resilience_score']:.3f}<br>
                Priority: #{fmt_rank(r['priority_rank'])}/132
            </div>
            """,
            max_width=220
        ),
        icon=folium.Icon(color="darkblue", icon="star", prefix="fa")
    ).add_to(m)

## 8. Add Legend

In [12]:
legend_html = """
<div style="
    position: fixed;
    bottom: 40px; left: 40px;
    background: white;
    padding: 15px 18px;
    border-radius: 10px;
    border: 1px solid #ddd;
    font-family: Arial;
    font-size: 13px;
    z-index: 9999;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.15);
">
    <b style="font-size:14px;">Station Type</b><br>
    <span style="color:#2196F3; font-size:18px;">&#9679;</span>
        Commuter<br>
    <span style="color:#FFC107; font-size:18px;">&#9679;</span>
        Mixed<br>
    <span style="color:#F44336; font-size:18px;">&#9679;</span>
        Leisure<br>
    <hr style="margin:8px 0; border-color:#eee;">
    <small style="color:#666;">
        Circle size = avg daily cyclists<br>
        Click any station for details<br>
        Use layers (top right) to switch views
    </small>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

## 9. Add Layer Control & Save

In [13]:
folium.LayerControl(collapsed=False).add_to(m)

output_path = OUTPUT / "interactive_map.html"
m.save(str(output_path))
print(f"Interactive map saved: {output_path}")
print(f"Stations plotted: {len(stations)}")
print(f"Layers: Station type | Priority | Volume")
print(f"Open in browser to explore")

Interactive map saved: /Users/queena/Documents/KU Leuven/Study material/Semester 1.2/Modern Analysis/Project/mda-cycling-weather-group6/outputs/interactive_map.html
Stations plotted: 133
Layers: Station type | Priority | Volume
Open in browser to explore
